# 02 — Model Training Walkthrough
**Wake Word Detection & Intent Classification System**

This notebook demonstrates:
1. Training the CNN-GRU wake-word model (dry run)
2. Fine-tuning BERT for intent classification (dry run)
3. Evaluating both models
4. Running end-to-end pipeline demo

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import matplotlib.pyplot as plt

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 1. Wake Word Model — Dry Run Training

In [ ]:
from src.training.train_wake_word import train_wake_word

model, test_metrics = train_wake_word(
    config_path='../configs/training_config.yaml',
    model_config_path='../configs/model_config.yaml',
    dry_run=True,   # 100 samples, 1 epoch
)

print('\n=== Wake Word Test Metrics ===')
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f'  {k:<15}: {v:.4f}')

In [ ]:
# Model summary
from src.models.wake_word_model import WakeWordCNNGRU
model_demo = WakeWordCNNGRU(n_mfcc=40)
print(f'Model parameters: {model_demo.count_parameters():,}')

# Forward pass timing
import time
x = torch.randn(1, 40, 98)
times = []
for _ in range(50):
    t0 = time.perf_counter()
    model_demo(x)
    times.append((time.perf_counter() - t0) * 1000)

print(f'Forward pass latency (50 runs):')
print(f'  Mean: {np.mean(times):.2f} ms')
print(f'  P90 : {np.percentile(times, 90):.2f} ms')

## 2. Intent Classifier — Stub Mode Demo

> Full BERT fine-tuning requires running `src/training/train_intent.py`. 
> This cell demonstrates the stub classifier that works without trained weights.

In [ ]:
from src.inference.intent_classifier import IntentClassifier

clf = IntentClassifier(model_path=None)  # stub mode

test_utterances = [
    'play some jazz music',
    'set an alarm for 7 AM',
    'what is the weather like today',
    'navigate to the airport',
    'send a message to Alice',
    'tell me a joke',
    'convert 5 miles to kilometres',
    'what is my battery level',
]

print(f'{"Utterance":<42} {"Intent":<20} {"Confidence"}')
print('-' * 75)
for text in test_utterances:
    result = clf.classify(text)
    print(f'{text:<42} {result["intent"]:<20} {result["confidence"]:.3f}')

## 3. End-to-End Pipeline Demo

In [ ]:
from src.inference.pipeline import VoicePipeline
from src.inference.wake_word_detector import WakeWordDetector
from src.inference.stt_engine import STTEngine
from src.audio.audio_capture import AudioStreamer

# Build pipeline components (stub/random-init modes)
detector = WakeWordDetector(model_path=None, threshold=0.1)  # low threshold to trigger
stt = STTEngine()
classifier = IntentClassifier(model_path=None)

pipeline = VoicePipeline(
    wake_word_detector=detector,
    stt_engine=stt,
    intent_classifier=classifier,
    latency_budget_ms=100,
)

# Simulate a 1-second audio chunk (1 kHz tone)
SR = 16_000
t = np.linspace(0, 1.0, SR, endpoint=False)
demo_audio = (0.5 * np.sin(2 * np.pi * 1000 * t)).astype(np.float32)

result = pipeline.process_chunk(demo_audio)

print('=== Pipeline Result ===')
print(f'Wake detected  : {result["wake_detected"]}')
print(f'Wake confidence: {result.get("wake_confidence", 0):.4f}')
if result.get('wake_detected'):
    print(f'Transcription  : "{result.get("transcription", "")}"')
    print(f'Intent         : {result.get("intent", "unknown")}')
    print(f'Confidence     : {result.get("intent_confidence", 0):.4f}')
print(f'Total latency  : {result.get("total_ms", 0):.2f} ms')
print(f'Within budget  : {result.get("within_budget", False)}')